In [45]:
import pandas as pd
import geopandas as gpd


In [46]:
raw_data_path = "../data/cdmx/raw/"
curated_data_path = "../data/cdmx/curated/"
geodata_file = curated_data_path + "geodata.gpkg"

In [47]:
cdmx_crs = "EPSG:32614" # UTM-14

## Metropolitan Zone
Filter raw data to obtain just municipalities from the metropolitan area of ​​the valley of mexico

In [48]:
metropolitan_zones = gpd.read_file(raw_data_path + "zm_scii/Zonas_Metropolitanas.shp", 
                                   encoding="latin-1")
# Leave only ZMVM and save in poligons gopecakge file
# if geopackage "geodata.gpkg" does not exist, it will be created. If it exists, the layer "zmvm" will be replaced.
metropolitan_zones.query("ID_ZM == '01'", inplace=True)
metropolitan_zones.to_file(geodata_file, 
                           layer="zmvm", 
                           driver="GPKG")


## Catalogs (stations, parameters, units)
Convert stations to gospatial data


### Stations

In [49]:
stations = pd.read_csv(raw_data_path + "cat_stations.csv", 
                       encoding="latin-1", 
                       skiprows=1, 
                       delimiter=";")

In [ ]:
stations.rename(columns={"cve_estac": "key_station",
                         "nom_estac": "name_station",
                         "latitud": "lat",
                         "longitud": "long",
                         "obs_estac": "status"}, 
                inplace=True)
stations['status'] = stations['status'].fillna('active')
stations.to_csv(curated_data_path + "cat_stations.csv", index=False)

In [51]:
geo_stations = gpd.GeoDataFrame(stations, 
                                geometry=gpd.points_from_xy(stations.long, stations.lat), 
                                crs="EPSG:4326")
geo_stations.to_crs(cdmx_crs, inplace=True)
geo_stations.to_file(geodata_file, 
                     layer="stations", 
                     driver="GPKG")


### Parameters

In [56]:
params = pd.read_csv(raw_data_path + "cat_params.csv", 
                     encoding="latin-1", 
                     skiprows=1, 
                     delimiter=";")

In [60]:
params.rename(columns={"id_parameter": "id_param",
                       "cve_param": "key_param",
                       "nom_param": "name_param",
                       "unidades_param": "units"}, 
              inplace=True)

params.to_csv(curated_data_path + "cat_params.csv", index=False)

### Units

In [61]:
units = pd.read_csv(raw_data_path + "cat_units.csv",
                    encoding="latin-1",
                    delimiter=";")

In [64]:
units.rename(columns={"id_unidad": "id_unit",
                      "cve_unidad": "key_unit",
                      "nom_unidad": "name_unit"},
             inplace=True)

units.to_csv(curated_data_path + "cat_units.csv", index=False)